# Checkpoint 4D — Time-Window Sensitivity (NOAA-19, January 2024, `mep_omni_flux_p1`)

**Question:** *How stable is the threshold-defined candidate high-flux footprint as the aggregation
time window changes* (1 day -> 7 -> 14 -> full month, plus four disjoint weeks)? Satellite, channel,
region, grid logic and threshold machinery are all **fixed**; only the time window varies.

This is **time-window sensitivity**, not a final South Atlantic Anomaly boundary. One-day maps are
expected to be orbit-track sparse. Coverage (orbital sampling) effects are reported separately from
possible physical/temporal variability. No dose / health-risk / danger / discovery claims.

- Satellite: NOAA-19 · Channel: `mep_omni_flux_p1` (differential proton flux ~25 MeV, `#/cm2-s-str-MeV`)
- Region: lat[-70,20] x lon[-100,20]; longitude converted [0,360) -> [-180,180)
- Calibration: drop `mep_IFC_on == 1`; keep `== -1` (uninterpreted)
- Grids: 5deg, 2deg · Statistics: mean_flux, median_flux · Thresholds: top 20/10/5/2/1%
- **Per-window coverage threshold** anchored to CP4A: `max(3, round((30/31)*day_count))`
  (so the full month reproduces CP4A's `>=30`; shorter windows relax proportionally).

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from saa.time_window_analysis import (
    TIME_WINDOWS, CUMULATIVE_LABELS, WEEK_LABELS,
    load_month_region, slice_window, window_day_count, choose_coverage_threshold,
    build_window_grids, sample_count_distribution, run_time_window_sensitivity, save_sensitivity,
    plot_window_mean_map, plot_mean_map_panels, plot_sample_count_panels,
    plot_centroid_by_window, plot_area_by_window, plot_weekly_centroids,
)
from saa.threshold_analysis import haversine_km

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TBL = ROOT / "outputs" / "tables"
FIG = ROOT / "outputs" / "figures"
for d in (PROC, TBL, FIG):
    d.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)

ROOT: /Users/fellipegoncalvesleite/saa-poes-mapping


## 1. Load the whole month once (real files), then slice each window

In [2]:
region, loaded_dates = load_month_region(raw_dir=str(RAW))
loaded_set = {pd.Timestamp(d).normalize() for d in loaded_dates}
print(f"month region rows: {len(region):,}  | daily files loaded: {len(loaded_dates)}/31")
print("date span:", region['obs_date'].min().date(), "->", region['obs_date'].max().date())
# honest per-window file accounting
def window_files(start, end):
    days = pd.date_range(start, end, freq="D")
    exp = len(days)
    load = sum(pd.Timestamp(d).normalize() in loaded_set for d in days)
    missing = [str(pd.Timestamp(d).date()) for d in days if pd.Timestamp(d).normalize() not in loaded_set]
    return exp, load, missing

month region rows: 205,153  | daily files loaded: 31/31
date span: 2024-01-01 -> 2024-01-31


## 2. Build per-window region slices + 5deg/2deg grids; report sample-count distributions & chosen coverage thresholds

In [3]:
specs = []
report_rows = []
window_tables = {}   # window_label -> {5: df5, 2: df2}
for w in TIME_WINDOWS:
    dc = window_day_count(w["start_date"], w["end_date"])
    sub = slice_window(region, w["start_date"], w["end_date"])
    grids, thr = build_window_grids(sub, dc)
    exp, load, missing = window_files(w["start_date"], w["end_date"])
    window_tables[w["window_label"]] = {g["grid_deg"]: g["table"] for g in grids}
    for g in grids:
        dist = sample_count_distribution(g["table"])
        cov = int(g["table"][g["mask_col"]].sum())
        report_rows.append({
            "window": w["window_label"], "day_count": dc, "grid_deg": g["grid_deg"],
            "rows": len(sub), "coverage_threshold": thr,
            "cells_populated": dist["n_cells"], "cells_pass_coverage": cov,
            "sc_min": dist["min"], "sc_median": dist["median"], "sc_max": dist["max"],
            "files_expected": exp, "files_loaded": load, "missing": ",".join(missing) or "none",
        })
    specs.append(dict(window_label=w["window_label"], start_date=w["start_date"], end_date=w["end_date"],
                      day_count=dc, files_expected=exp, files_loaded=load, grids=grids))

report = pd.DataFrame(report_rows)
pd.set_option("display.width", 200, "display.max_columns", 30)
print(report.to_string(index=False))

               window  day_count  grid_deg   rows  coverage_threshold  cells_populated  cells_pass_coverage  sc_min  sc_median  sc_max  files_expected  files_loaded missing
       day_2024-01-01          1         5   7244                   3              206                  201       1       42.0      86               1             1    none
       day_2024-01-01          1         2   7244                   3              548                  517       1       16.0      36               1             1    none
days_2024-01-01_to_07          7         5  43524                   7              429                  429       9      102.0     172               7             7    none
days_2024-01-01_to_07          7         2  43524                   7             2298                 2097       1       17.0      36               7             7    none
days_2024-01-01_to_14         14         5  88277                  14              432                  432      48      208.0     331 

**Coverage thresholds chosen** (per window, same for both grids as in CP4A): day=3, 7-day/weekly=7,
14-day=14, month=30. These are *not* a blind reuse of CP4A's `>=30` — they scale with exposure
(`round((30/31)*day_count)`, floored at 3) so a 1-day map is not silently held to a 31-day standard.
The full month row reproduces CP4A coverage exactly (432/432 at 5deg, 2685/2700 at 2deg), confirming
consistency with the accepted product.

## 3. Save processed regional subsets (4 cumulative windows) + 16 grid tables

In [4]:
# regional parquets for the 4 cumulative windows
cum_paths = {
    "day_2024-01-01":        PROC / "cp4d_noaa19_2024-01-01_mep_omni_flux_p1_region.parquet",
    "days_2024-01-01_to_07": PROC / "cp4d_noaa19_2024-01-01_to_07_mep_omni_flux_p1_region.parquet",
    "days_2024-01-01_to_14": PROC / "cp4d_noaa19_2024-01-01_to_14_mep_omni_flux_p1_region.parquet",
    "month_2024-01":         PROC / "cp4d_noaa19_2024-01_full_month_mep_omni_flux_p1_region.parquet",
}
for w in TIME_WINDOWS:
    if w["window_label"] in cum_paths:
        sub = slice_window(region, w["start_date"], w["end_date"])
        sub.to_parquet(cum_paths[w["window_label"]], index=False)
        print(f"saved {cum_paths[w['window_label']].name}: {len(sub):,} rows")

# all 16 grid tables
for label, tabs in window_tables.items():
    for gd, tab in tabs.items():
        p = TBL / f"cp4d_{label}_grid_{gd}deg.parquet"
        tab.to_parquet(p, index=False)
print("grid tables written:", len(window_tables) * 2)

saved cp4d_noaa19_2024-01-01_mep_omni_flux_p1_region.parquet: 7,244 rows
saved cp4d_noaa19_2024-01-01_to_07_mep_omni_flux_p1_region.parquet: 43,524 rows
saved cp4d_noaa19_2024-01-01_to_14_mep_omni_flux_p1_region.parquet: 88,277 rows
saved cp4d_noaa19_2024-01_full_month_mep_omni_flux_p1_region.parquet: 205,153 rows


grid tables written: 16


## 4. Run threshold sensitivity (8 windows x 2 grids x 2 stats x 5 thresholds = 160 rows)

In [5]:
sens = run_time_window_sensitivity(specs)
save_sensitivity(sens, TBL / "cp4d_time_window_threshold_sensitivity.csv",
                 TBL / "cp4d_time_window_threshold_sensitivity.parquet")
print("rows:", len(sens), "| windows:", sens.window_label.nunique(),
      "| selected_cell_count min:", int(sens.selected_cell_count.min()))
print("rows with a coverage_warning:", int((sens.coverage_warning.str.len() > 0).sum()))
sens.head(10)

rows: 160 | windows: 8 | selected_cell_count min: 3
rows with a coverage_warning: 20


,window_label,start_date,end_date,day_count,files_expected,files_loaded,grid_deg,statistic_used,threshold_label,percentile_cutoff,flux_cutoff_value,cells_available_after_coverage_mask,selected_cell_count,selected_area_km2,selected_area_fraction_of_covered_region,centroid_lat_unweighted,centroid_lon_unweighted,centroid_lat_flux_weighted,centroid_lon_flux_weighted,peak_flux,mean_flux_within_selected,median_flux_within_selected,total_flux_area_proxy,coverage_threshold_used,coverage_warning
0,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,mean_flux,top20,80,3.080779,201,41,1.188904e+07,0.243402,-17.134146,-55.060976,-17.806842,-54.828556,33.137061,12.328031,11.044499,1.465239e+08,3,one-day orbit-track sparse; low_coverage:201/4...
1,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,mean_flux,top10,90,11.044499,201,21,6.110104e+06,0.125091,-18.214286,-53.690476,-18.594564,-54.174315,33.137061,18.208419,15.347249,1.109449e+08,3,one-day orbit-track sparse; low_coverage:201/4...
2,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,mean_flux,top5,95,15.347249,201,11,3.185034e+06,0.065207,-19.318182,-55.227273,-19.576100,-55.071888,33.137061,22.939183,23.028570,7.290942e+07,3,one-day orbit-track sparse; low_coverage:201/4...
3,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,mean_flux,top2,98,23.584470,201,5,1.441463e+06,0.029511,-20.500000,-54.500000,-20.637745,-54.450427,33.137061,29.017160,30.193611,4.176585e+07,3,one-day orbit-track sparse; low_coverage:201/4...
4,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,mean_flux,top1,99,30.193611,201,3,8.656833e+05,0.017723,-20.833333,-54.166667,-20.856917,-54.110073,33.137061,31.291074,30.542551,2.707804e+07,3,one-day orbit-track sparse; low_coverage:201/4...
5,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,median_flux,top20,80,2.856200,201,41,1.188904e+07,0.243402,-17.134146,-55.060976,-17.810345,-54.630629,34.477200,12.266154,10.541050,1.457948e+08,3,one-day orbit-track sparse; low_coverage:201/4...
6,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,median_flux,top10,90,10.541050,201,21,6.110104e+06,0.125091,-18.214286,-53.690476,-18.626574,-54.049212,34.477200,18.303340,15.218900,1.114945e+08,3,one-day orbit-track sparse; low_coverage:201/4...
7,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,median_flux,top5,95,15.218900,201,11,3.173376e+06,0.064968,-20.227273,-57.500000,-20.290895,-56.568360,34.477200,23.185927,22.823000,7.347732e+07,3,one-day orbit-track sparse; low_coverage:201/4...
8,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,median_flux,top2,98,23.341499,201,5,1.441463e+06,0.029511,-20.500000,-54.500000,-20.680837,-54.445492,34.477200,29.405120,30.421200,4.231444e+07,3,one-day orbit-track sparse; low_coverage:201/4...
9,day_2024-01-01,2024-01-01,2024-01-01,1,1,1,5,median_flux,top1,99,30.421200,201,3,8.656833e+05,0.017723,-20.833333,-54.166667,-20.876963,-54.095077,34.477200,31.920366,30.862700,2.761911e+07,3,one-day orbit-track sparse; low_coverage:201/4...


In [6]:
# compact view: top10, 5deg mean across all windows
view = sens[(sens.grid_deg == 5) & (sens.statistic_used == "mean_flux") & (sens.threshold_label == "top10")]
print(view[["window_label","day_count","cells_available_after_coverage_mask","selected_cell_count",
            "selected_area_km2","centroid_lat_flux_weighted","centroid_lon_flux_weighted","peak_flux",
            "coverage_warning"]].to_string(index=False))

         window_label  day_count  cells_available_after_coverage_mask  selected_cell_count  selected_area_km2  centroid_lat_flux_weighted  centroid_lon_flux_weighted  peak_flux                                             coverage_warning
       day_2024-01-01          1                                  201                   21       6.110104e+06                  -18.594564                  -54.174315  33.137061 one-day orbit-track sparse; low_coverage:201/432 cells (47%)
days_2024-01-01_to_07          7                                  429                   43       1.234489e+07                  -20.553322                  -55.592608  31.233597                                                             
days_2024-01-01_to_14         14                                  432                   44       1.261403e+07                  -20.859383                  -55.439200  31.570600                                                             
        month_2024-01         31                

## 5. Centroid stability: how much does the footprint center move as the window grows?

In [7]:
def fw_centroid(label, thr="top10", gd=5, stat="mean_flux"):
    r = sens[(sens.window_label==label)&(sens.grid_deg==gd)&(sens.statistic_used==stat)&(sens.threshold_label==thr)].iloc[0]
    return r.centroid_lat_flux_weighted, r.centroid_lon_flux_weighted

for thr in ("top10", "top5"):
    print(f"\n== flux-weighted centroid drift, {thr}, 5deg mean ==")
    chain = CUMULATIVE_LABELS
    prev = None
    for lbl in chain:
        lat, lon = fw_centroid(lbl, thr)
        d = "" if prev is None else f"  (+{haversine_km(prev[0],prev[1],lat,lon):6.0f} km vs prev)"
        print(f"  {lbl:24s} ({lat:6.2f}, {lon:7.2f}){d}")
        prev = (lat, lon)
    d_lat, d_lon = fw_centroid('day_2024-01-01', thr)
    m_lat, m_lon = fw_centroid('month_2024-01', thr)
    print(f"  day -> month total shift: {haversine_km(d_lat,d_lon,m_lat,m_lon):.0f} km")


== flux-weighted centroid drift, top10, 5deg mean ==
  day_2024-01-01           (-18.59,  -54.17)
  days_2024-01-01_to_07    (-20.55,  -55.59)  (+   264 km vs prev)
  days_2024-01-01_to_14    (-20.86,  -55.44)  (+    38 km vs prev)
  month_2024-01            (-20.89,  -55.48)  (+     5 km vs prev)
  day -> month total shift: 289 km

== flux-weighted centroid drift, top5, 5deg mean ==
  day_2024-01-01           (-19.58,  -55.07)
  days_2024-01-01_to_07    (-20.91,  -55.21)  (+   149 km vs prev)
  days_2024-01-01_to_14    (-21.44,  -55.02)  (+    62 km vs prev)
  month_2024-01            (-21.43,  -55.64)  (+    64 km vs prev)
  day -> month total shift: 215 km


## 6. Weekly stability (four disjoint weeks) + 5deg-vs-2deg and mean-vs-median agreement

In [8]:
print("== weekly flux-weighted centroids (top10, 5deg mean) ==")
wk = [(l, *fw_centroid(l, "top10")) for l in WEEK_LABELS]
for l, lat, lon in wk:
    print(f"  {l}: ({lat:6.2f}, {lon:7.2f})")
import itertools
maxd = max(haversine_km(a[1],a[2],b[1],b[2]) for a,b in itertools.combinations(wk,2))
print(f"  max pairwise weekly centroid distance: {maxd:.0f} km")

print("\n== 5deg vs 2deg agreement (month, top10, mean) flux-weighted centroid ==")
c5 = fw_centroid("month_2024-01","top10",5); c2 = fw_centroid("month_2024-01","top10",2)
print(f"  5deg ({c5[0]:.2f},{c5[1]:.2f}) vs 2deg ({c2[0]:.2f},{c2[1]:.2f}) -> {haversine_km(*c5,*c2):.0f} km")

print("\n== mean vs median agreement (month, top10, 5deg) flux-weighted centroid ==")
cm = fw_centroid("month_2024-01","top10",5,"mean_flux"); cd = fw_centroid("month_2024-01","top10",5,"median_flux")
print(f"  mean ({cm[0]:.2f},{cm[1]:.2f}) vs median ({cd[0]:.2f},{cd[1]:.2f}) -> {haversine_km(*cm,*cd):.0f} km")

print("\n== windows flagged low-coverage / sparse ==")
flagged = sorted(set(sens.loc[sens.coverage_warning.str.len()>0, "window_label"]))
print("  ", flagged or "none")

== weekly flux-weighted centroids (top10, 5deg mean) ==
  week1: (-20.55,  -55.59)
  week2: (-21.51,  -55.10)
  week3: (-20.66,  -55.46)
  week4: (-20.50,  -55.45)
  max pairwise weekly centroid distance: 118 km

== 5deg vs 2deg agreement (month, top10, mean) flux-weighted centroid ==
  5deg (-20.89,-55.48) vs 2deg (-20.63,-55.61) -> 31 km

== mean vs median agreement (month, top10, 5deg) flux-weighted centroid ==
  mean (-20.89,-55.48) vs median (-20.86,-55.46) -> 4 km

== windows flagged low-coverage / sparse ==
   ['day_2024-01-01']


## 7. Figures (no smoothing / no interpolation; blank = no data or below coverage threshold)

In [9]:
# A. cumulative mean maps, 5deg
amap = {
    "day_2024-01-01":        "cp4d_mean_flux_5deg_day_2024-01-01.png",
    "days_2024-01-01_to_07": "cp4d_mean_flux_5deg_days_2024-01-01_to_07.png",
    "days_2024-01-01_to_14": "cp4d_mean_flux_5deg_days_2024-01-01_to_14.png",
    "month_2024-01":         "cp4d_mean_flux_5deg_month_2024-01.png",
}
for lbl, fname in amap.items():
    tab = window_tables[lbl][5]
    note = " (orbit-track sparse)" if lbl == "day_2024-01-01" else ""
    plot_window_mean_map(tab, 5.0, "enough_samples_5deg", FIG / fname,
                         f"mean flux 5deg - {lbl}{note}\nNOAA-19 p1 (~25 MeV); exploratory; not a final SAA boundary")
    print("saved", fname)

# B. sample-count comparison panels (5deg and 2deg)
cum_tab5 = [(l, window_tables[l][5]) for l in CUMULATIVE_LABELS]
cum_tab2 = [(l, window_tables[l][2]) for l in CUMULATIVE_LABELS]
plot_sample_count_panels(cum_tab5, 5.0, FIG / "cp4d_sample_count_5deg_time_windows.png",
                         "sample_count by time window - 5deg (coverage growth)")
plot_sample_count_panels(cum_tab2, 2.0, FIG / "cp4d_sample_count_2deg_time_windows.png",
                         "sample_count by time window - 2deg (coverage growth)")
print("saved sample-count panels")

saved cp4d_mean_flux_5deg_day_2024-01-01.png


saved cp4d_mean_flux_5deg_days_2024-01-01_to_07.png
saved cp4d_mean_flux_5deg_days_2024-01-01_to_14.png


saved cp4d_mean_flux_5deg_month_2024-01.png


saved sample-count panels


In [10]:
# C. centroid stability (top10, top5) across cumulative windows
plot_centroid_by_window(sens, "top10", FIG / "cp4d_centroid_by_time_window_top10.png", labels=CUMULATIVE_LABELS)
plot_centroid_by_window(sens, "top5",  FIG / "cp4d_centroid_by_time_window_top5.png",  labels=CUMULATIVE_LABELS)
# D. area stability
plot_area_by_window(sens, FIG / "cp4d_area_by_time_window.png", labels=CUMULATIVE_LABELS)
# E. weekly comparison
plot_weekly_centroids(sens, FIG / "cp4d_weekly_centroid_comparison.png")
week_tab5 = [(l, window_tables[l][5]) for l in WEEK_LABELS]
plot_mean_map_panels(week_tab5, 5.0, "enough_samples_5deg", FIG / "cp4d_weekly_mean_flux_5deg_comparison.png",
                     "weekly mean flux 5deg (disjoint weeks) - NOAA-19 p1; exploratory")
print("saved centroid/area/weekly figures")
print("all CP4D figures:", sorted(p.name for p in FIG.glob("cp4d_*.png")))

saved centroid/area/weekly figures
all CP4D figures: ['cp4d_area_by_time_window.png', 'cp4d_centroid_by_time_window_top10.png', 'cp4d_centroid_by_time_window_top5.png', 'cp4d_mean_flux_5deg_day_2024-01-01.png', 'cp4d_mean_flux_5deg_days_2024-01-01_to_07.png', 'cp4d_mean_flux_5deg_days_2024-01-01_to_14.png', 'cp4d_mean_flux_5deg_month_2024-01.png', 'cp4d_sample_count_2deg_time_windows.png', 'cp4d_sample_count_5deg_time_windows.png', 'cp4d_weekly_centroid_comparison.png', 'cp4d_weekly_mean_flux_5deg_comparison.png']


## 8. Summary

- **Coverage is the dominant difference between windows, and it is measured.** The one-day map covers
  only ~47% (5deg) / ~19% (2deg) of region cells and is flagged `one-day orbit-track sparse`; by 7 days
  coverage is near-complete at 5deg, and the full month reproduces CP4A coverage exactly.
- **Centroid stability:** the flux-weighted center of the top-10%/top-5% footprint moves most from the
  (sparse) 1-day window to 7 days, then changes little from 7 -> 14 -> month — i.e. the candidate
  footprint stabilises once coverage fills in. Quantified in section 5.
- **Weekly windows** (disjoint, equal exposure) cluster tightly (section 6), so the residual day->month
  drift is largely a coverage/sampling effect, not a large week-to-week physical swing.
- **5deg vs 2deg** and **mean vs median** agree qualitatively (centers within section-6 distances).
- **Too sparse to trust as-is:** the 1-day window (especially 2deg, tightest thresholds) — reported but
  flagged, never treated as final.

This is **time-window sensitivity of a threshold-defined, coverage-limited footprint** — *not* a true
SAA boundary, center, dose, health risk, danger zone, or discovery. `mep_IFC_on == -1` retained,
uninterpreted.